# FrugalProver pilot — three independent tiers

Runs on a Colab **T4 GPU** (`Runtime > Change runtime type > T4`).

The prover/solving layer is NOT being changed here -- this notebook only pulls
activations `g_ell(x)` from a small model, and *separately, optionally*
collects budget/outcome signal at different costs. Pick ONE `RUN_MODE` per
run (see the config cell):

| RUN_MODE | What it does | Cost | Input file |
|---|---|---|---|
| `activations_only` | phi(x) + g_ell(x) at EVERY layer. No generation at all. | ~free, scales to hundreds of problems | `math_pool_large_partN.json` |
| `full_sweep` | The expensive part: full budget sweep (all BUDGETS x N_SAMPLES), gives real B*/p(B) with a reported confidence interval | expensive, keep n small | `math_labeling_subset_partN.json` |
| `baseline_single` | ONE fixed budget x a few samples -- a cheap, coarse solved/not-solved signal at larger scale than full_sweep | cheap-ish, scales further than full_sweep | `math_pool_large_partN.json` (or a chunk of it) |

`math_labeling_subset.json`'s problem ids are a NESTED subset of
`math_pool_large.json`'s ids (see `data_prep.py`), so `merge_results.py` can
join activations + full_sweep + baseline_single records by id into one
record per problem, whichever tiers happened to cover it.

Output: `pilot_{RUN_MODE}_{RUN_TAG}.jsonl`, checkpointed after every problem
so a Colab disconnect doesn't lose progress.

In [ ]:
!pip -q install transformers accelerate

In [ ]:
from google.colab import files
print('Upload the file matching your RUN_MODE (see the table above), e.g. math_pool_large_part1.json')
uploaded = files.upload()

In [ ]:
import json, re, time
from collections import Counter
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- per-person knobs: set these before running, then leave the rest alone ---
RUN_MODE = "activations_only"   # "activations_only" | "full_sweep" | "baseline_single"
RUN_TAG = "part1"               # "part1"/"part2"/"part3" -- tags the output filename
SUBSET_FILE = "math_pool_large_part1.json"  # match RUN_MODE per the table above
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # a SMALL model is fine/better here (see markdown)
POOLING = "mean"                # "mean" or "last" -- how g_ell(x) is pooled over prompt tokens

# --- only matter for full_sweep / baseline_single (ignored in activations_only) ---
N_PROBLEMS = 300         # slice of SUBSET_FILE actually run this session
BUDGETS = [128, 256, 512]  # full_sweep: max_new_tokens caps to sweep
N_SAMPLES = 3             # full_sweep: samples per (problem, budget)
SUCCESS_THRESHOLD = 0.5   # p(B) >= threshold defines "solved at this budget"
BASELINE_BUDGET = 256     # baseline_single: the one budget level tested
BASELINE_SAMPLES = 3      # baseline_single: samples at that one budget
GEN_BATCH_SIZE = 6        # sequences generated in parallel per batch

# RUN A 2-3 PROBLEM DRY RUN FIRST (set N_PROBLEMS=3 above) to measure real
# per-problem seconds on this GPU, then extrapolate and reset N_PROBLEMS.

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="cuda"
)
model.eval()
n_layers = model.config.num_hidden_layers
# Pull ALL layers, not just a few fixed depths: it's the SAME forward pass
# either way (output_hidden_states=True already computes every layer), so
# keeping all of them costs ~nothing and lets analyze.py/oracle.py do a real
# layer-depth sweep (PIPELINE.md 2.1 risk #1: "choice of layer is not trivial").
layer_idx = list(range(n_layers + 1))  # 0 = embeddings, n_layers = final hidden state
print(f"[{RUN_MODE}/{RUN_TAG}] model={MODEL_NAME} has {n_layers} layers; "
      f"sampling activations at ALL {len(layer_idx)} layers, pooling={POOLING}")

In [ ]:
with open(SUBSET_FILE, encoding="utf-8") as f:
    problems = json.load(f)[:N_PROBLEMS]
print(f"[{RUN_MODE}/{RUN_TAG}] loaded {len(problems)} problems from {SUBSET_FILE}")

In [ ]:
# --- phi(x): cheap surface / "external" features -- the deconfounding baseline
# from PIPELINE.md 2.1 p.2. `type` (subject) is folded in separately in analyze.py
# since it comes from MATH metadata, not the raw text.

def max_brace_depth(s: str) -> int:
    depth = maxd = 0
    for ch in s:
        if ch == "{":
            depth += 1
            maxd = max(maxd, depth)
        elif ch == "}":
            depth = max(0, depth - 1)
    return maxd

def surface_features(problem: str) -> dict:
    return {
        "char_len": len(problem),
        "word_len": len(problem.split()),
        "latex_cmd_count": len(re.findall(r"\\[a-zA-Z]+", problem)),
        "dollar_count": problem.count("$"),
        "has_asy_figure": "[asy]" in problem,
        "brace_depth": max_brace_depth(problem),
        "digit_count": sum(ch.isdigit() for ch in problem),
        "eq_count": problem.count("="),
    }

In [ ]:
# --- g_ell(x): forward-pass-only activation extraction. POOLING is the adapter:
# "mean" averages hidden states over all prompt tokens, "last" takes only the
# final token's hidden state. This is the ONLY thing activations_only mode runs
# -- no generation, so it's cheap regardless of how weak/strong MODEL_NAME is. ---

@torch.no_grad()
def extract_activations(problem: str) -> dict:
    prompt = f"Problem:\n{problem}\n\nSolution:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model(**inputs, output_hidden_states=True)
    feats = {}
    for li in layer_idx:
        h = out.hidden_states[li][0]  # (seq_len, hidden)
        pooled = h[-1] if POOLING == "last" else h.mean(dim=0)
        feats[f"layer_{li}"] = pooled.float().cpu().tolist()
    return feats

In [ ]:
# --- answer extraction + grading, and a Wilson score CI for p(B) estimates --
# with only N_SAMPLES~3 trials per budget, a bare point estimate for p(B) is
# not "adequate" on its own; report the CI so everyone sees how much noise is
# in each B* label, instead of treating B* as exact. ---

BOXED_RE = re.compile(r"\\boxed\{")

def extract_boxed(text: str):
    matches = list(BOXED_RE.finditer(text))
    if not matches:
        return None
    start = matches[-1].end()
    depth, i = 1, start
    while i < len(text) and depth > 0:
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
        i += 1
    return text[start : i - 1].strip()

def normalize(ans: str) -> str:
    if ans is None:
        return ""
    a = ans.strip()
    a = a.replace("\\left", "").replace("\\right", "")
    a = a.replace(" ", "").replace("\\!", "")
    a = a.replace("\\dfrac", "\\frac").replace("\\tfrac", "\\frac")
    a = a.rstrip(".")
    return a

def extract_answer(generated_text: str):
    """Best-effort extraction of the model's final answer, normalized. None if not found."""
    pred = extract_boxed(generated_text)
    if pred is None:
        m = re.search(r"final answer is[:\s]*\$?([^\n$]+)\$?", generated_text, re.IGNORECASE)
        pred = m.group(1) if m else None
    norm = normalize(pred)
    return norm if norm != "" else None

def wilson_ci(successes: int, n: int, z: float = 1.96):
    """Wilson score interval for a binomial proportion -- honest uncertainty
    for p(B) given only n=N_SAMPLES trials, instead of a bare point estimate."""
    if n == 0:
        return (0.0, 1.0)
    phat = successes / n
    denom = 1 + z * z / n
    center = (phat + z * z / (2 * n)) / denom
    margin = (z * ((phat * (1 - phat) / n + z * z / (4 * n * n)) ** 0.5)) / denom
    return (max(0.0, center - margin), min(1.0, center + margin))

In [ ]:
# --- full_sweep: the expensive tier. From the SAME samples we still get both
# p_by_budget (independent-attempt accuracy) and sc_by_budget (self-consistency
# majority vote) for free, plus a Wilson CI per budget for how noisy each
# p(B) point actually is (the "adequate estimate" ask). ---

SOLVE_PROMPT = (
    "Solve the following competition math problem. Show brief reasoning, "
    "then end with a line 'Final answer: \\boxed{...}'.\n\nProblem:\n{problem}\n\nSolution:"
)

@torch.no_grad()
def _generate_answers(prompt, budget, n_samples):
    preds = []
    for batch_start in range(0, n_samples, GEN_BATCH_SIZE):
        bs = min(GEN_BATCH_SIZE, n_samples - batch_start)
        inputs = tokenizer([prompt] * bs, return_tensors="pt", padding=True).to(model.device)
        out = model.generate(
            **inputs, max_new_tokens=budget, do_sample=True, temperature=0.7, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
        gen_only = out[:, inputs["input_ids"].shape[1]:]
        texts = tokenizer.batch_decode(gen_only, skip_special_tokens=True)
        preds.extend(extract_answer(t) for t in texts)
    return preds

def run_budget_sweep(problem: str, gold_answer: str) -> dict:
    prompt = SOLVE_PROMPT.format(problem=problem)
    gold_norm = normalize(gold_answer)
    p_by_budget, p_ci_by_budget, sc_by_budget = {}, {}, {}
    for budget in BUDGETS:
        preds = _generate_answers(prompt, budget, N_SAMPLES)
        successes = sum(p == gold_norm for p in preds)
        p_by_budget[budget] = successes / len(preds)
        p_ci_by_budget[budget] = list(wilson_ci(successes, len(preds)))

        non_null = [p for p in preds if p is not None]
        majority = Counter(non_null).most_common(1)[0][0] if non_null else None
        sc_by_budget[budget] = float(majority == gold_norm)

    solved_budgets = [b for b, p in p_by_budget.items() if p >= SUCCESS_THRESHOLD]
    b_star = min(solved_budgets) if solved_budgets else None
    return {"p_by_budget": p_by_budget, "p_ci_by_budget": p_ci_by_budget,
            "sc_by_budget": sc_by_budget, "b_star": b_star}

def run_baseline_single(problem: str, gold_answer: str) -> dict:
    """baseline_single tier: ONE budget, a few samples -- cheaper than the full
    sweep, so it can cover many more problems, at the cost of only telling you
    solved/not-solved at that one point, not a full B* estimate."""
    prompt = SOLVE_PROMPT.format(problem=problem)
    gold_norm = normalize(gold_answer)
    preds = _generate_answers(prompt, BASELINE_BUDGET, BASELINE_SAMPLES)
    successes = sum(p == gold_norm for p in preds)
    ci_lo, ci_hi = wilson_ci(successes, len(preds))
    non_null = [p for p in preds if p is not None]
    majority = Counter(non_null).most_common(1)[0][0] if non_null else None
    return {
        "baseline_budget": BASELINE_BUDGET,
        "p_baseline": successes / len(preds),
        "p_baseline_ci": [ci_lo, ci_hi],
        "sc_baseline": float(majority == gold_norm),
    }

In [ ]:
# --- main loop, checkpointed to disk after every problem. Output filename
# encodes BOTH the tier (RUN_MODE) and the part (RUN_TAG) so nothing collides
# and merge_results.py can tell tiers apart. ---

out_path = f"pilot_{RUN_MODE}_{RUN_TAG}.jsonl"
done_ids = set()
try:
    with open(out_path, encoding="utf-8") as f:
        for line in f:
            done_ids.add(json.loads(line)["id"])
    print(f"[{RUN_MODE}/{RUN_TAG}] resuming, {len(done_ids)} problems already done")
except FileNotFoundError:
    pass

with open(out_path, "a", encoding="utf-8") as f:
    for row in problems:
        if row["id"] in done_ids:
            continue
        t0 = time.time()
        phi = surface_features(row["problem"])
        acts = extract_activations(row["problem"])

        extra = {}
        if RUN_MODE == "full_sweep":
            extra = run_budget_sweep(row["problem"], row["answer"])
        elif RUN_MODE == "baseline_single":
            extra = run_baseline_single(row["problem"], row["answer"])
        # activations_only: extra stays {} -- no generation at all

        record = {
            "id": row["id"],
            "run_mode": RUN_MODE,
            "run_tag": RUN_TAG,
            "pooling": POOLING,
            "model_name": MODEL_NAME,
            "level_num": row["level_num"],
            "type": row["type"],
            "problem": row["problem"],
            "phi": phi,
            "activations": acts,
            **extra,
        }
        f.write(json.dumps(record) + "\n")
        f.flush()
        dt = time.time() - t0
        extra_str = ""
        if RUN_MODE == "full_sweep":
            extra_str = f" b_star={extra['b_star']} p={extra['p_by_budget']}"
        elif RUN_MODE == "baseline_single":
            extra_str = f" p@{BASELINE_BUDGET}={extra['p_baseline']:.2f} CI={extra['p_baseline_ci']}"
        print(f"[{RUN_MODE}/{RUN_TAG}] id={row['id']:>3} level={row['level_num']}{extra_str} "
              f"({dt:.1f}s, est. remaining {(dt * (len(problems) - row['id'] - 1))/60:.1f} min)")

In [ ]:
from google.colab import files
files.download(f"pilot_{RUN_MODE}_{RUN_TAG}.jsonl")